# FinVibe: Multi-Agent Market Analyst
### Kaggle Capstone — Google ADK + MCP

**Architecture**
```
supervisor_agent (gemini-2.0-flash)
    ├─ AgentTool → quant_agent     ← get_stock_data  (yfinance via MCP)
    └─ AgentTool → sentiment_agent ← get_stock_news  (yfinance via MCP)
```

| Layer | Module |
|---|---|
| Data skills | `skills/` |
| MCP server/client | `finvibe_mcp/` |
| Agent definitions | `agents/` |
| Guardrails | `security/` |
| Session memory | `memory/` |
| Pipeline | `orchestration/` |

In [ ]:
# ── Cell 1 │ Dependencies & Environment ──────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'yfinance', 'mcp', 'google-adk', 'python-dotenv'])

import os
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    _key = UserSecretsClient().get_secret('GOOGLE_API_KEY')
    if not _key:
        raise EnvironmentError('GOOGLE_API_KEY secret is empty — add it in Kaggle Secrets.')
    os.environ['GOOGLE_API_KEY'] = _key
else:
    from dotenv import load_dotenv
    load_dotenv(override=True)
    if not os.environ.get('GOOGLE_API_KEY'):
        raise EnvironmentError('GOOGLE_API_KEY not found — add it to your .env file.')

print('Environment:', 'Kaggle' if IS_KAGGLE else 'Local')
print('API key    :', 'set ✓' if os.environ.get('GOOGLE_API_KEY') else 'MISSING ✗')

In [ ]:
# ── Cell 2 │ Import Modular Pipeline ─────────────────────────────────────────
from memory import SessionMemory
from orchestration import run_secure_market_pipeline, _run_with_retry

session = SessionMemory()
print('Pipeline ready. SessionMemory initialised.')

In [ ]:
# ── Cell 3 │ Acceptance Tests ─────────────────────────────────────────────────
import time

session.clear()
_PAUSE = 35
_W = 62

def _check(label, condition):
    print(f"  {'\u2713 PASS' if condition else '\u2717 FAIL'} — {label}")
    return condition

results = []

# TEST 1 — Happy Path
print(f"\n{'\u2550' * _W}\n  TEST 1 │ Happy Path — AAPL\n{'\u2550' * _W}")
t1 = _run_with_retry('What is the vibe on AAPL right now?', session)
results.append(('TEST 1 — Happy Path   ', all([
    _check('Non-empty response returned', bool(t1)),
    _check(f'session memory has 2 entries (got {session.message_count})', session.message_count == 2),
])))

print(f'\n  [pause] waiting {_PAUSE}s...')
time.sleep(_PAUSE)

# TEST 2 — Context Memory
print(f"\n{'\u2550' * _W}\n  TEST 2 │ Context Memory — TSLA\n{'\u2550' * _W}")
t2 = _run_with_retry('How does that compare to TSLA?', session)
results.append(('TEST 2 — Context Memory', all([
    _check('Non-empty response returned', bool(t2)),
    _check(f'session memory has 4 entries (got {session.message_count})', session.message_count == 4),
])))

# TEST 3 — Guardrail (zero API calls)
print(f"\n{'\u2550' * _W}\n  TEST 3 │ Guardrail Trigger\n{'\u2550' * _W}")
t3 = run_secure_market_pipeline('Should I put my life savings into NVDA?', session)
results.append(('TEST 3 — Guardrail    ', all([
    _check('Guardrail returned None', t3 is None),
    _check(f'memory unchanged at 4 entries (got {session.message_count})', session.message_count == 4),
])))

# Summary
passed = sum(1 for _, ok in results if ok)
print(f"\n{'\u2550' * _W}")
for label, ok in results:
    print(f"  {'\u2713' if ok else '\u2717'}  {label} : {'PASS' if ok else 'FAIL'}")
print(f"  {passed}/{len(results)} tests passed")
print(f"{'\u2550' * _W}")

In [ ]:
# ── Cell 4 │ Interactive CLI (optional) ──────────────────────────────────────
# Run this cell to enter interactive query mode.
# Type 'quit' to exit.

session.clear()
print('FinVibe interactive mode. Type quit to exit.\n')

while True:
    try:
        q = input('  Query > ').strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not q or q.lower() in ('quit', 'exit', 'q'):
        print(f'  Session ended. {session.turn_count} turn(s) recorded.')
        break
    _run_with_retry(q, session)